## Library Imports & Environment Setup

This section imports the core Python libraries required for building and evaluating the machine learning classification models.  

- **NumPy & Pandas** → Data manipulation and numerical operations  
- **Scikit-learn** → Data preprocessing, model training, and evaluation  
- **XGBoost** → Advanced gradient boosting model for high performance  
- **Evaluation Metrics** → Accuracy, Precision, Recall, F1-Score, and ROC-AUC for model assessment  

Warnings are suppressed to ensure a clean and readable output during model experimentation.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler,LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.svm  import SVC
from xgboost import XGBClassifier

from sklearn.metrics import (accuracy_score,
                             precision_score,
                             recall_score,
                             f1_score,
                             roc_auc_score
                             )


import warnings
warnings.filterwarnings('ignore')

### Load Cleaned Dataset

In this step, the preprocessed shooting dataset is loaded into a Pandas DataFrame.  
This dataset contains cleaned and structured records that will be used for further analysis, feature engineering, and machine learning model development.

`pandas.read_csv()` is used to import the dataset, and `df.head()` is displayed to preview the first few records and verify that the data has been loaded correctly.

In [ ]:
df=pd.read_csv("/content/cleaned_shooting_data.csv")
df.head()


,PRECINCT,JURISDICTION_CODE,LOCATION_DESC,STATISTICAL_MURDER_FLAG,PERP_AGE_GROUP,PERP_SEX,PERP_RACE,VIC_AGE_GROUP,VIC_RACE,Latitude,Longitude,YEAR,MONTH,DAY_OF_WEEK,HOUR
0,105,0.0,Unknown,False,Unknown,Unknown,Unknown,18-24,BLACK,40.662965,-73.730839,2021,5,3,21
1,40,0.0,Unknown,False,Unknown,Unknown,Unknown,18-24,BLACK,40.810352,-73.924942,2014,6,4,17
2,108,0.0,Unknown,True,Unknown,Unknown,Unknown,25-44,WHITE,40.742607,-73.915492,2015,11,5,3
3,44,0.0,Unknown,False,Unknown,Unknown,Unknown,<18,WHITE HISPANIC,40.837782,-73.919457,2015,10,4,18
4,47,0.0,Unknown,True,25-44,M,BLACK,45-64,BLACK,40.886238,-73.852910,2009,2,3,22


In [ ]:
df.columns

Index(['PRECINCT', 'JURISDICTION_CODE', 'LOCATION_DESC',
       'STATISTICAL_MURDER_FLAG', 'PERP_AGE_GROUP', 'PERP_SEX', 'PERP_RACE',
       'VIC_AGE_GROUP', 'VIC_RACE', 'Latitude', 'Longitude', 'YEAR', 'MONTH',
       'DAY_OF_WEEK', 'HOUR'],
      dtype='object')

In [ ]:
print("Initial Shape:", df.shape)

Initial Shape: (27308, 15)


### Feature Engineering

In this step, domain-driven features are created to capture important temporal and demographic patterns that may influence crime severity and fatality risk.

Key engineered features include:

- **IS_NIGHT**: Identifies incidents occurring during night hours (8 PM – 4 AM), which may correlate with reduced visibility and higher risk conditions.
- **IS_WEEKEND**: Flags incidents occurring on weekends when traffic behavior and activity patterns differ from weekdays.
- **IS_SUMMER**: Captures seasonal patterns by identifying incidents during summer months (June–August).
- **IS_ELDERLY_VICTIM**: Indicates whether the victim belongs to the elderly age group (65+), representing a potentially vulnerable population.
- **IS_ELDERLY_PERP**: Identifies cases where the perpetrator belongs to the elderly age group (65+).
- **NIGHT_WEEKEND**: Interaction feature combining night-time and weekend conditions to capture high-risk temporal scenarios.

These engineered features help the model better learn **behavioral, temporal, and demographic patterns**, improving predictive performance for fatality risk classification.

In [ ]:
df['IS_NIGHT'] = df['HOUR'].apply(lambda x: 1 if (x >= 20 or x <= 4) else 0)

df['IS_WEEKEND'] = df['DAY_OF_WEEK'].apply(lambda x: 1 if x >= 5 else 0)

df['IS_SUMMER'] = df['MONTH'].apply(lambda x: 1 if x in [6,7,8] else 0)

df['IS_ELDERLY_VICTIM'] = df['VIC_AGE_GROUP'].apply(lambda x: 1 if x == '65+' else 0)

df['IS_ELDERLY_PERP'] = df['PERP_AGE_GROUP'].apply(lambda x: 1 if x == '65+' else 0)

df['NIGHT_WEEKEND'] = df['IS_NIGHT'] * df['IS_WEEKEND']

print("Feature Engineering Done:", df.shape)


print("Feature Engineering Completed. Current Shape:", df.shape)



Feature Engineering Done: (27308, 21)
Feature Engineering Completed. Current Shape: (27308, 21)


### Categorical Feature Encoding

To prepare categorical variables for machine learning models, two encoding strategies were applied based on the **cardinality of the features**.

**1. High-Cardinality Encoding (Frequency Encoding)**  
Features such as `PRECINCT` and `LOCATION_DESC` contain a large number of unique categories. Applying One-Hot Encoding on these variables would significantly increase the dimensionality of the dataset.  
To address this, **Frequency Encoding** was used, where each category is replaced with its occurrence frequency in the dataset. This preserves useful information while keeping the feature space compact.

**2. Low-Cardinality Encoding (One-Hot Encoding)**  
Categorical features with relatively fewer unique values — such as demographic attributes (`PERP_AGE_GROUP`, `PERP_SEX`, `PERP_RACE`, `VIC_AGE_GROUP`, `VIC_RACE`) and `JURISDICTION_CODE` — were transformed using **One-Hot Encoding**.  
This converts each category into binary indicator variables, enabling machine learning models to effectively interpret categorical information.

Finally, the original high-cardinality columns were removed to avoid redundancy, and the dataset was updated with the encoded features.

In [ ]:
high_card_cols = ['PRECINCT', 'LOCATION_DESC']

for col in high_card_cols:
    freq = df[col].value_counts() / len(df)
    df[col + '_FREQ'] = df[col].map(freq)

# Drop original high-card columns
df = df.drop(columns=high_card_cols)

print("High Card Encoding Done:", df.shape)

low_card_cols = [
    'PERP_AGE_GROUP',
    'PERP_SEX',
    'PERP_RACE',
    'VIC_AGE_GROUP',
    'VIC_RACE',
    'JURISDICTION_CODE'
]

df = pd.get_dummies(df, columns=low_card_cols, drop_first=True)

print("Low Card Encoding Done:", df.shape)

High Card Encoding Done: (27308, 21)
Low Card Encoding Done: (27308, 48)


In [ ]:
list(df.columns)

['STATISTICAL_MURDER_FLAG',
 'Latitude',
 'Longitude',
 'YEAR',
 'MONTH',
 'DAY_OF_WEEK',
 'HOUR',
 'IS_NIGHT',
 'IS_WEEKEND',
 'IS_SUMMER',
 'IS_ELDERLY_VICTIM',
 'IS_ELDERLY_PERP',
 'NIGHT_WEEKEND',
 'PRECINCT_FREQ',
 'LOCATION_DESC_FREQ',
 'PERP_AGE_GROUP_18-24',
 'PERP_AGE_GROUP_25-44',
 'PERP_AGE_GROUP_45-64',
 'PERP_AGE_GROUP_65+',
 'PERP_AGE_GROUP_<18',
 'PERP_AGE_GROUP_UNKNOWN',
 'PERP_AGE_GROUP_Unknown',
 'PERP_SEX_F',
 'PERP_SEX_M',
 'PERP_SEX_U',
 'PERP_SEX_Unknown',
 'PERP_RACE_AMERICAN INDIAN/ALASKAN NATIVE',
 'PERP_RACE_ASIAN / PACIFIC ISLANDER',
 'PERP_RACE_BLACK',
 'PERP_RACE_BLACK HISPANIC',
 'PERP_RACE_UNKNOWN',
 'PERP_RACE_Unknown',
 'PERP_RACE_WHITE',
 'PERP_RACE_WHITE HISPANIC',
 'VIC_AGE_GROUP_25-44',
 'VIC_AGE_GROUP_45-64',
 'VIC_AGE_GROUP_65+',
 'VIC_AGE_GROUP_<18',
 'VIC_AGE_GROUP_UNKNOWN',
 'VIC_RACE_ASIAN / PACIFIC ISLANDER',
 'VIC_RACE_BLACK',
 'VIC_RACE_BLACK HISPANIC',
 'VIC_RACE_UNKNOWN',
 'VIC_RACE_WHITE',
 'VIC_RACE_WHITE HISPANIC',
 'JURISDICTION_CODE_

### Feature–Target Separation

The dataset is prepared for model training by separating the **target variable** from the feature set.

- **Target Variable (`y`)**: `STATISTICAL_MURDER_FLAG` – indicates whether the incident is classified as a murder case.
- **Feature Set (`X`)**: All remaining columns used as predictors.

Additionally, a **sanity check** is performed to:
- Verify the final shape of the feature matrix.
- Ensure no unintended **object (categorical) columns** remain before model training.

This step ensures the dataset is correctly structured and ready for machine learning algorithms.

In [ ]:
X = df.drop(columns=['STATISTICAL_MURDER_FLAG'])
y = df['STATISTICAL_MURDER_FLAG']

print("Final Feature Shape:", X.shape)

# Sanity check
print("Object columns remaining:",
      X.select_dtypes(include=['object']).columns)

Final Feature Shape: (27308, 47)
Object columns remaining: Index([], dtype='object')


In [ ]:
print("Selected Columns:")
print(X.columns)

Selected Columns:
Index(['Latitude', 'Longitude', 'YEAR', 'MONTH', 'DAY_OF_WEEK', 'HOUR',
       'IS_NIGHT', 'IS_WEEKEND', 'IS_SUMMER', 'IS_ELDERLY_VICTIM',
       'IS_ELDERLY_PERP', 'NIGHT_WEEKEND', 'PRECINCT_FREQ',
       'LOCATION_DESC_FREQ', 'PERP_AGE_GROUP_18-24', 'PERP_AGE_GROUP_25-44',
       'PERP_AGE_GROUP_45-64', 'PERP_AGE_GROUP_65+', 'PERP_AGE_GROUP_<18',
       'PERP_AGE_GROUP_UNKNOWN', 'PERP_AGE_GROUP_Unknown', 'PERP_SEX_F',
       'PERP_SEX_M', 'PERP_SEX_U', 'PERP_SEX_Unknown',
       'PERP_RACE_AMERICAN INDIAN/ALASKAN NATIVE',
       'PERP_RACE_ASIAN / PACIFIC ISLANDER', 'PERP_RACE_BLACK',
       'PERP_RACE_BLACK HISPANIC', 'PERP_RACE_UNKNOWN', 'PERP_RACE_Unknown',
       'PERP_RACE_WHITE', 'PERP_RACE_WHITE HISPANIC', 'VIC_AGE_GROUP_25-44',
       'VIC_AGE_GROUP_45-64', 'VIC_AGE_GROUP_65+', 'VIC_AGE_GROUP_<18',
       'VIC_AGE_GROUP_UNKNOWN', 'VIC_RACE_ASIAN / PACIFIC ISLANDER',
       'VIC_RACE_BLACK', 'VIC_RACE_BLACK HISPANIC', 'VIC_RACE_UNKNOWN',
       'VIC_RACE_WHIT

### Feature Name Sanitization

To ensure compatibility with machine learning libraries (such as XGBoost) and maintain consistent feature naming conventions, column names are sanitized by removing or replacing special characters.

This step replaces symbols such as `<`, `>`, `/`, and spaces with standardized text or underscores, and removes any remaining non-alphanumeric characters. The result is a clean, uniform set of feature names that prevents parsing errors and improves pipeline reliability.

In [ ]:
X.columns = (
    X.columns
    .str.replace('<', 'less_', regex=False)
    .str.replace('>', 'greater_', regex=False)
    .str.replace('/', '_', regex=False)
    .str.replace(' ', '_', regex=False)
    .str.replace('[^A-Za-z0-9_]', '', regex=True)
)

### Train–Test Split

The dataset is divided into training and testing sets to evaluate the model's performance on unseen data.

- **Test Size:** 20% of the dataset is reserved for testing.
- **Random State:** `42` is used to ensure reproducibility of results.
- **Stratification:** `stratify=y` maintains the same class distribution in both training and testing sets.

This approach helps prevent data leakage and ensures reliable model evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("X Shape:", X.shape)
print("y Shape:", y.shape)
print("-" * 40)

print("X_train Shape:", X_train.shape)
print("X_test Shape:", X_test.shape)
print("y_train Shape:", y_train.shape)
print("y_test Shape:", y_test.shape)

X Shape: (27308, 47)
y Shape: (27308,)
----------------------------------------
X_train Shape: (21846, 47)
X_test Shape: (5462, 47)
y_train Shape: (21846,)
y_test Shape: (5462,)


In [ ]:
print(X_train.dtypes[X_train.dtypes == 'object'])

Series([], dtype: object)


### Feature Scaling (Standardization)

To ensure consistent feature magnitudes and improve model performance, **StandardScaler** is applied to the continuous numerical features.  
The scaler standardizes each feature by removing the mean and scaling to unit variance.

- The scaler is **fit only on the training data** to prevent data leakage.
- The same transformation is then **applied to the test data**.

Scaled Features:
- Latitude
- Longitude
- Year
- Month
- Day of Week
- Hour

This step helps machine learning models converge faster and ensures that features with larger scales do not dominate the learning process.

In [ ]:
scaler = StandardScaler()

# Define only continuous numerical columns manually
num_cols = [
    'Latitude',
    'Longitude',
    'YEAR',
    'MONTH',
    'DAY_OF_WEEK',
    'HOUR'
]

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

### Model Evaluation with Stratified Cross-Validation

This function evaluates classification models using **Stratified K-Fold Cross Validation** and multiple performance metrics to ensure robust and reliable model assessment.

Key evaluation steps:

- **Stratified K-Fold (5 folds)** is used to maintain class balance during cross-validation.
- **F1 Score** is used as the primary cross-validation metric to balance precision and recall.
- The model is trained on the training dataset and evaluated on the test dataset.
- Predictions and probability scores are generated for performance analysis.

The following evaluation metrics are calculated:

- **Accuracy** – Overall prediction correctness  
- **Precision** – Proportion of correct positive predictions  
- **Recall** – Ability to detect actual positive cases  
- **F1 Score** – Harmonic mean of precision and recall  
- **ROC-AUC Score** – Model’s ability to distinguish between classes

This structured evaluation approach ensures **reliable model comparison and selection for the final predictive model.**

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test):

    # -----------------------------
    # Cross Validation
    # -----------------------------
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=skf,
        scoring='f1'
    )

    # Train Model
    model.fit(X_train, y_train)


    # Predictions

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = model.decision_function(X_test)


    # Metrics

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)


    # Print Results

    print("="*50)
    print("Model:", model.__class__.__name__)
    print("-"*50)
    print("CV F1 Score (Mean):", np.round(cv_scores.mean(), 4))
    print("Test Accuracy:", np.round(acc, 4))
    print("Test Precision:", np.round(prec, 4))
    print("Test Recall:", np.round(rec, 4))
    print("Test F1 Score:", np.round(f1, 4))
    print("Test ROC-AUC:", np.round(roc, 4))

    print("="*50)

    return {
        "Model": model.__class__.__name__,
        "CV_F1": cv_scores.mean(),
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": roc
    }

## Model Selection and Benchmarking

To identify the most effective model for cardiovascular risk prediction, multiple machine learning algorithms were evaluated. The selected models represent a mix of linear, tree-based, ensemble, and boosting techniques commonly used in classification tasks.

Class imbalance in the dataset was addressed using **class weighting** for most models and **scale_pos_weight** in XGBoost to ensure balanced learning between minority and majority classes.

The following models were included in the comparison:

- Logistic Regression
- Support Vector Machine (SVM)
- Decision Tree Classifier
- Random Forest Classifier
- Gradient Boosting Classifier
- XGBoost Classifier

Each model is initialized with controlled parameters such as `random_state` for reproducibility and configured to handle class imbalance effectively. These models will be trained and evaluated using cross-validation and classification metrics such as **Accuracy, Precision, Recall, F1-Score, and ROC-AUC** to determine the best-performing algorithm for the final deployment.

In [ ]:
models = [

    LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ),

    SVC(
        class_weight='balanced',
        probability=True,
        random_state=42
    ),

    DecisionTreeClassifier(
        criterion='gini',
        class_weight='balanced',
        random_state=42
    ),

    RandomForestClassifier(
        class_weight='balanced',
        random_state=42
    ),

    GradientBoostingClassifier(
        random_state=42
    ),

    XGBClassifier(
        use_label_encoder=False,
        eval_metric='logloss',
        scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1]),
        random_state=42
    )
]

### Model Evaluation and Performance Comparison

In this step, multiple machine learning models are evaluated using the previously defined `evaluate_model()` function.  
Each model is trained on the training dataset and tested on the unseen test dataset.

The evaluation metrics include:
- **Accuracy**
- **Precision**
- **Recall**
- **F1 Score**

All model results are stored in a list and converted into a pandas DataFrame for easier comparison.  
Finally, the models are sorted based on **F1 Score** to identify the best-performing model for the classification task.

This comparison helps in selecting the most reliable model for the final cardiovascular risk prediction system.

In [ ]:
results = []

for model in models:
    result = evaluate_model(
        model,
        X_train,
        y_train,
        X_test,
        y_test
    )
    results.append(result)

results_df = pd.DataFrame(results)
print("\nFinal Model Comparison:")
print(results_df.sort_values(by="F1", ascending=False))

Model: LogisticRegression
--------------------------------------------------
CV F1 Score (Mean): 0.3823
Test Accuracy: 0.5932
Test Precision: 0.2638
Test Recall: 0.6201
Test F1 Score: 0.3702
Test ROC-AUC: 0.6523
Model: SVC
--------------------------------------------------
CV F1 Score (Mean): 0.3908
Test Accuracy: 0.5981
Test Precision: 0.2747
Test Recall: 0.661
Test F1 Score: 0.3881
Test ROC-AUC: 0.6667
Model: DecisionTreeClassifier
--------------------------------------------------
CV F1 Score (Mean): 0.2849
Test Accuracy: 0.7073
Test Precision: 0.2655
Test Recall: 0.2934
Test F1 Score: 0.2788
Test ROC-AUC: 0.5476
Model: RandomForestClassifier
--------------------------------------------------
CV F1 Score (Mean): 0.1193
Test Accuracy: 0.7706
Test Precision: 0.2207
Test Recall: 0.075
Test F1 Score: 0.112
Test ROC-AUC: 0.6382
Model: GradientBoostingClassifier
--------------------------------------------------
CV F1 Score (Mean): 0.0531
Test Accuracy: 0.8101
Test Precision: 0.7
Test Rec

In [ ]:
df.to_csv("processed_shooting_data.csv", index=False)